# Linear resistive stability analysis with RDCON

An equilibrium that DCON calls ideally stable can still tear. Finite
resistivity lets the field reconnect at rational surfaces, and whether an
island there grows or heals is set by the tearing stability index
**Delta-prime**: positive and the island grows, negative and it decays.

RDCON computes Delta-prime for every rational surface a given toroidal mode
number has inside the plasma, using the same outer-region solution DCON does
and matching it across the resistive layer. This notebook runs it on an
equilibrium stored in the repository, reads the per-surface matching data, maps
what has a correct IMAS home into `mhd_linear` and `ntms`, and draws the
registered plots.

Read `linear_ideal_stability_analysis_with_dcon.ipynb` first if you have not:
the resistive case starts from the same prepared equilibrium and the same
namelist templates, and the ideal result is the context for this one.

## 0. Setup

RDCON is part of the GPEC suite, which VAFT does not ship and cannot install.
VAFT resolves `$GPECHOME/bin/rdcon`, the same `$XHOME` convention the other
external codes follow; `bin/rmatch` is its companion. Without them every
section below reports what it would show and stops.

Everything else is in the repository: the `rdcon.in`, `rmatch.in`, `equil.in`
and `vac.in` templates in `vaft/data/gpec`, and the equilibrium
`vaft/data/efit/g039915.00319`. The run goes into a temporary directory and
section 7 removes it.

The packaged namelists are used unmodified -- no reduced grid, no shortened
mode set beyond the toroidal modes chosen below.

In [ ]:
import contextlib
import os
import tempfile
import time
from pathlib import Path

try:
    _ipython = get_ipython()
except NameError:
    _ipython = None
if _ipython is None:
    os.environ.setdefault("MPLBACKEND", "Agg")
elif "IPKernelApp" in _ipython.config:
    _ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np

import vaft
from vaft.code import gpec
from vaft.code._executables import executable_from_home, missing_home_message
from vaft.data.resources import data_path

plt.rcParams["figure.dpi"] = 110

SHOT = 39915
TIME_MS = 319
GEQDSK = data_path(f"efit/g0{SHOT}.00{TIME_MS}")

# The adapter's default mode set. RDCON is markedly more expensive than DCON
# per mode -- n=1 takes about 47 s against DCON's 8 s on this equilibrium, and
# n=3 ran for over twenty minutes without finishing -- so raise this
# deliberately, not by habit.
MODES = (1, 2)

# DCON runs alongside RDCON, for two reasons. The ideal verdict is the context
# the resistive one needs -- "tearing stable" means little if the equilibrium is
# already ideally unstable -- and the eigenfunction the plot catalog draws comes
# from DCON's `solutions.bin`, which RDCON does not write. See section 4.
CONFIG = gpec.GPECSuiteConfig(modules=("dcon", "rdcon"), modes=MODES)
EXECUTABLE = executable_from_home(
    os.environ.get(gpec.GPEC_HOME_ENV),
    home_variable=gpec.GPEC_HOME_ENV,
    relative_path="bin/rdcon",
    code_name="RDCON",
)
HAVE_RDCON = EXECUTABLE is not None

print(f"equilibrium {GEQDSK.name}  ({'found' if GEQDSK.is_file() else 'MISSING'})")
print(f"templates   {data_path('gpec')}")
print(f"modes       {', '.join(f'n={n}' for n in MODES)}")
print(f"executable  {EXECUTABLE}")
if not HAVE_RDCON:
    print()
    print(missing_home_message(
        home_variable=gpec.GPEC_HOME_ENV,
        relative_path="bin/rdcon",
        code_name="RDCON",
    ))
    print("\nThe GPEC-suite build itself is tracked in issue #226.")

## 1. Prepare and run

Identical staging to the ideal case: one directory per (time, module, mode)
under `{workdir}/{time}/{module}/nn={n}/`, each holding the equilibrium and its
namelists.

RDCON writes the Delta-prime this notebook reads. `rmatch` is a **companion**
that runs afterwards in the same directory and adds the global solution
(`globalsol.bin`); it is not what produces Delta-prime. The adapter reports one
status for the whole cell, so a cell can read `failed` because the companion
failed while RDCON itself terminated normally -- which is why the check below
asks the solver whether its own output is usable instead of reading the
aggregate return code.

That distinction is not hypothetical: `globalsol.bin` is missing from every
cell below. `rmatch` reads `eta` and `massden` from `&RMATCH_INPUT` as one
value *per rational surface*, and the packaged `rmatch.in` supplies a single
scalar, so it stops with

```
ERROR: eta requires  11 non-zero elements
```

-- eleven for `n=1` here, twenty for `n=2`. The count depends on the
equilibrium and the toroidal mode, so no fixed value in a shared template can
satisfy it; supplying the resistivity profile is a per-case decision that this
notebook does not make for you. Delta-prime is unaffected, because RDCON
computes it and `rmatch` does not.

Check `rmatch.log` in the run directory whenever a cell reports `failed` or a
companion output is missing.

In [ ]:
_cleanup = contextlib.ExitStack()
WORKDIR = None
result = None

if HAVE_RDCON:
    WORKDIR = Path(
        _cleanup.enter_context(tempfile.TemporaryDirectory(prefix="vaft-rdcon-"))
    )
    inputs = gpec.GPECCaseInputs(
        shot=SHOT, time_ms=TIME_MS, geqdsk=GEQDSK, workdir=WORKDIR
    )
    print(f"work directory  {WORKDIR}")

    _started = time.monotonic()
    result = gpec.run_gpec_suite_case(inputs, CONFIG)
    print(f"elapsed         {time.monotonic() - _started:.0f} s")
    print()
    for record in result.records:
        # A cell's status folds in its optional companion, so it can read
        # `failed` while the solver itself terminated normally and wrote the
        # output this notebook needs. Ask each solver's own success check what
        # is actually usable rather than trusting the aggregate return code.
        usable, why = gpec.SOLVERS[record.module].check_success(
            record.workdir, record.mode
        )
        print(f"  {record.module:5s} n={record.mode}  cell {record.status:9s}"
              f" rc={record.returncode}  ->  output"
              f" {'USABLE' if usable else 'unusable'}")
        if not usable and why:
            print(f"      {why}")
        if record.reason:
            # The adapter's own explanation. A timeout reports returncode None
            # and puts its reason only here, so a run that ran out of time is
            # not mistaken for one that failed on its inputs.
            print(f"      {record.reason}")
        elif record.missing_optional_outputs:
            print(f"      companion did not write: "
                  f"{', '.join(record.missing_optional_outputs)}")

    USABLE_MODES = tuple(sorted({
        record.mode for record in result.records
        if record.module == "rdcon"
        and gpec.SOLVERS["rdcon"].check_success(record.workdir, record.mode)[0]
    }))
    IDEAL_MODES = tuple(sorted({
        record.mode for record in result.records
        if record.module == "dcon"
        and gpec.SOLVERS["dcon"].check_success(record.workdir, record.mode)[0]
    }))
else:
    USABLE_MODES = ()
    IDEAL_MODES = ()

HAVE_RUN = bool(USABLE_MODES)
if not HAVE_RDCON:
    print(
        "Would prepare one directory per toroidal mode, run RDCON and its\n"
        "matching companion in each, and report the status of every cell."
    )

## 2. Delta-prime, surface by surface

This is what the run exists to produce. For each toroidal mode, RDCON locates
the rational surfaces where `q = m/n` and reports the matching data there.

`Delta_prime` is a **matrix**, not a vector: entry `(i, j)` couples surface `i`
to surface `j`. The classical, single-surface tearing index is its diagonal,
and that is the only part with an IMAS home. `delta_prime_diagonal()` extracts
it -- the same call the IMAS mapping makes, so the table below and `ntms` in
section 3 cannot disagree. The off-diagonal coupling stays in the native
container.

The sign convention is the standard one: **Delta-prime > 0 is tearing
unstable** at that surface, and more positive means a faster-growing island.
Negative is stable. The imaginary part is not a stability statement -- it
carries the rotation and the asymmetry of the matching -- and IMAS `ntms` has
no slot for it, so it stays in the native container.

**Not every surface in the table is interpretable, and this equilibrium makes
that unusually obvious.** Two effects dominate the extremes:

- **Very close to the axis** the asymptotic matching's thin-layer assumption
  fails, and Delta-prime blows up by ten orders of magnitude. The `m=2, n=1`
  surface at `psi_n` of 0.08 below is a resolution artefact, not a violent
  instability.
- **Very close to the edge** the surfaces bunch. VEST's `q` runs to 12.3 at the
  boundary, so a dozen rational surfaces sit between `psi_n` 0.9 and 1.0, each
  with high `m`. Classical Delta-prime grows with `m` anyway, and the outer
  solution is least resolved exactly there, so the large positive values at the
  top of each list are not a prediction that VEST had twenty tearing modes.

What *is* interpretable is the low-`m` surfaces well inside the plasma. Nothing
is filtered out of the table -- the whole matrix diagonal is printed, and the
reader is told which part to read.

`q` at each surface should equal `m/n` exactly; it is printed as a check on the
surface-finding, not as a result.

In [ ]:
matching = {}

# The band in which a classical Delta-prime from an outer-region code is worth
# reading for this equilibrium: inside it the layer is resolved and the
# surfaces are well separated. The bounds are a judgement about *this* case --
# they are not a property of RDCON -- so they are named here rather than buried
# in a comparison, and nothing outside them is hidden, only labelled.
_INTERIOR = (0.2, 0.9)

if HAVE_RUN:
    for record in result.records:
        if record.module != "rdcon" or record.mode not in USABLE_MODES:
            continue
        native = gpec.read_pest3_matching_output(
            record.workdir, solver="rdcon", mode=record.mode
        )
        surfaces = native.delta_prime_diagonal()
        matching[record.mode] = surfaces
        if not surfaces:
            print(f"n={record.mode}: no rational surfaces inside the plasma")
            continue

        print(f"n={record.mode}:  {len(surfaces)} rational surfaces")
        print(f"   {'m':>3} {'psi_n':>8} {'q':>7} {'m/n':>7}"
              f" {'Re Delta-prime':>16}   verdict")
        for surface in surfaces:
            dp = surface["delta_prime_real"]
            interpretable = _INTERIOR[0] <= surface["psi_n"] <= _INTERIOR[1]
            if not interpretable:
                note = "near axis" if surface["psi_n"] < _INTERIOR[0] else "near edge"
                verdict = f"({note} -- see above)"
            else:
                verdict = "tearing UNSTABLE" if dp > 0 else "tearing stable"
            print(f"   {surface['m']:>3} {surface['psi_n']:>8.4f}"
                  f" {surface['q']:>7.3f} {surface['m'] / record.mode:>7.3f}"
                  f" {dp:>16.4g}   {verdict}")
        print()
else:
    print(
        "Would list every rational surface, its Delta-prime, and whether that\n"
        "surface is tearing stable."
    )

## 3. Into IMAS

The resistive result splits across two IDSs, and the split is deliberate rather
than incidental.

`mhd_linear` gets `n_tor` and `ballooning_type = "Tearing"` -- a correct
mode-type tag, and nothing more, because **Delta-prime has no field anywhere in
`mhd_linear`**. Tagging the mode as tearing must not be confused with storing
its stability index.

`ntms` gets the value, because the classical single-surface Delta-prime *is* a
legitimate `deltaw` contribution to the Rutherford equation. One entry per
rational surface, named `classical`, carrying the real part -- `deltaw.value`
is a real scalar, so the imaginary part has no home and stays in the native
container rather than being dropped quietly or coerced.

The full per-surface matching data, including the off-diagonal `A_prime`,
`B_prime` and `Gamma_prime` blocks, comes back from the mapper as a return
value for the run manifest, since none of it has an IMAS slot either.

In [ ]:
ods = None
manifest = None

if HAVE_RUN:
    from vaft.omas.vest_upstream import build_mhd_linear_ods

    ods, manifest = build_mhd_linear_ods(
        shot=SHOT, time_values=[TIME_MS], workdir=WORKDIR,
        modules=("dcon", "rdcon"), modes=tuple(sorted(set(USABLE_MODES + IDEAL_MODES))),
    )
    print(f"manifest status: {manifest['status']}")

    written = sorted(ods.flat())
    for ids in ("mhd_linear.time_slice", "ntms.time_slice"):
        paths = [p for p in written if p.startswith(ids)]
        print(f"\n{len(paths)} {ids} paths, e.g.")
        for path in paths[:5]:
            print(f"  {path}")

    print("\nntms deltaw contributions:")
    for mode in ods["ntms"]["time_slice"][0]["mode"]:
        entry = ods["ntms"]["time_slice"][0]["mode"][mode]
        value = entry["deltaw"][0]["value"]
        print(f"  m={int(entry['m_pol'])} n={int(entry['n_tor'])}"
              f"  {entry['deltaw'][0]['name']} = {value:+.4g}")
else:
    print("Would map the run into mhd_linear and ntms and list what was written.")

## 4. The registered plots

Three of the four registered `mhd_linear` recipes draw an eigenfunction, and
**that eigenfunction is DCON's, not RDCON's.** RDCON writes no `solutions.bin`
-- only a DCON run with its `match` companion does -- so a mapped RDCON-only
ODS carries `n_tor` and the `Tearing` mode tag but nothing for these recipes to
plot, and asking for one raises rather than drawing an empty axis. That is why
section 0 runs DCON alongside.

So the figures below are the ideal eigenfunction of the same equilibrium,
included because they are the spatial context for where the rational surfaces
in section 2 sit -- not because RDCON produced them. The resistive result's own
figure is the Delta-prime plot that follows.

In [ ]:
if HAVE_RUN and ods is not None:
    import vaft.omas as vomas

    figure, axes = vomas.plot_mhd_linear_profile_displacement(ods)
    axes.set_title(f"{axes.get_title()}  --  VEST {SHOT} @ {TIME_MS} ms (RDCON)")
    plt.show()
else:
    print("Would draw the displacement eigenfunction, one trace per harmonic.")

In [ ]:
if HAVE_RUN and ods is not None:
    figure, axes = vomas.plot_mhd_linear_profile_b_field_perturbed(ods)
    plt.show()
else:
    print("Would draw the perturbed field profile for the same mode.")

In [ ]:
if HAVE_RUN and ods is not None:
    figure, axes = vomas.plot_mhd_linear_overview_eigenfunction(ods)
    plt.show()
else:
    print("Would draw the overview panel for the least-stable mapped mode.")

Delta-prime itself has no registered recipe -- there is no `ntms` plot in the
catalog today -- so the surface-by-surface picture is drawn directly here. It
is the one figure a resistive run needs that the ideal one does not.

Every surface is plotted, on a symmetric-log axis so the near-axis artefact and
the interior values share one figure without either being dropped. The shaded
band marks where the result is worth reading; the points outside it are shown
because the run produced them, not because they mean anything.

In [ ]:
if HAVE_RUN and matching:
    figure, axes = plt.subplots(figsize=(6.5, 4.0))
    for mode, surfaces in sorted(matching.items()):
        if not surfaces:
            continue
        axes.plot([s["psi_n"] for s in surfaces],
                  [s["delta_prime_real"] for s in surfaces], "o-", label=f"n={mode}")
        for s in surfaces:
            if _INTERIOR[0] <= s["psi_n"] <= _INTERIOR[1]:
                axes.annotate(f"m={s['m']}", (s["psi_n"], s["delta_prime_real"]),
                              textcoords="offset points", xytext=(5, 5), fontsize=8)
    # Symlog keeps every surface on one axis -- the near-axis artefact spans
    # fourteen decades -- so nothing has to be dropped to make the figure
    # readable. The shaded band is the interpretable region, not a data cut.
    axes.set_yscale("symlog", linthresh=1.0)
    axes.axhline(0.0, color="0.5", linewidth=0.8)
    axes.axvspan(*_INTERIOR, color="0.92", zorder=0)
    axes.annotate("interpretable", (sum(_INTERIOR) / 2, 0.97), xycoords=("data", "axes fraction"),
                  ha="center", va="top", fontsize=8, color="0.35")
    axes.set_xlabel(r"$\psi_N$ of the rational surface")
    axes.set_ylabel(r"Re $\Delta'$   (symlog)")
    axes.set_title(f"Classical tearing index, VEST {SHOT} @ {TIME_MS} ms"
                   "\nabove zero is unstable; outside the band, resolution dominates")
    axes.legend()
    figure.tight_layout()
    plt.show()
else:
    print(
        "Would draw Delta-prime against the flux coordinate of each rational\n"
        "surface, one trace per toroidal mode, with the marginal line at zero."
    )

## 5. What this run establishes

**It does establish** the classical tearing stability index at each rational
surface of this reconstructed equilibrium, for the modes solved.

**It does not establish** whether an island actually grows on VEST.
Delta-prime is only the outer-region drive. The Rutherford equation balances it
against neoclassical, polarization and curvature terms that this calculation
does not compute, and a neoclassical tearing mode can grow at a surface with
Delta-prime < 0 given a seed island. `ntms.deltaw` is named for exactly that
sum, and what is written here is one contribution to it, labelled `classical`.

**It is only as good as the equilibrium.** Delta-prime depends on the current
profile gradient at the rational surface, which an EFIT reconstruction
constrains weakly. Refining the equilibrium first -- see
`equilibrium_refinement_using_chease.ipynb` -- changes these numbers.

**Only the interior surfaces are trustworthy.** Section 2 marks the rest. Near
the axis the matching assumption fails; near the edge, where VEST's `q` climbs
to 12.3 and a dozen high-`m` surfaces bunch into the last tenth of `psi_n`, the
values are dominated by resolution and by the `m`-scaling of Delta-prime
itself. Read the low-`m` interior surfaces and treat the rest as diagnostics of
the calculation rather than of the plasma.

## 6. Cleanup

The run leaves the outer-region solution and matching intermediates behind,
which are large and regenerable. The inputs are in the repository, so
re-running this notebook reproduces everything. Comment out the `close()` to
keep a run for inspection.

In [ ]:
if WORKDIR is not None and WORKDIR.exists():
    _size_mb = sum(p.stat().st_size for p in WORKDIR.rglob("*") if p.is_file()) / 1e6
    print(f"releasing {WORKDIR} ({_size_mb:.0f} MB)")
_cleanup.close()
print("done" if WORKDIR is None else f"removed: {not WORKDIR.exists()}")